# Day 2 notebook companion

Run in order with synthetic data. Mermaid diagrams render on the website. Setup and shared helpers are embedded; no checkout is required. Learner exercises report NOT ATTEMPTED until implemented. Reference checks are separate. Optional controls also have direct function calls.


In [ ]:
import importlib.metadata
import subprocess
import sys
for package, version in {"cryptography": "50.0.1", "matplotlib": "3.10.6", "ipywidgets": "8.1.7"}.items():
    try:
        installed = importlib.metadata.version(package)
    except importlib.metadata.PackageNotFoundError:
        installed = None
    if installed != version:
        subprocess.check_call([sys.executable, "-m", "pip", "install", f"{package}=={version}"])
print("Dependencies ready. Restart if an older library was already imported, then run all cells.")


## Shared teaching helpers

Inspect this implementation. TLS uses real SSL objects over memory buffers and temporary test key files; no system trust changes or network listeners. The teaching KDF is not a standardized protocol key schedule.


In [ ]:
"""Day 2 teaching helpers. Real TLS over MemoryBIO; no sockets or trust-store changes."""
from datetime import datetime, timedelta, timezone
from pathlib import Path
import ssl
import tempfile
import hashlib
from cryptography import x509
from cryptography.x509.oid import NameOID, ExtendedKeyUsageOID
from cryptography.hazmat.primitives import hashes, serialization
from cryptography.hazmat.primitives.asymmetric import ec
from cryptography.hazmat.primitives.kdf.hkdf import HKDF


def make_pki(expired=False):
    """Create an isolated root, intermediate, server, and client for this run."""
    now = datetime.now(timezone.utc)
    keys = {name: ec.generate_private_key(ec.SECP256R1())
            for name in ('root', 'intermediate', 'server', 'client')}
    names = {name: x509.Name([x509.NameAttribute(NameOID.COMMON_NAME, 'Workshop ' + name)])
             for name in keys}
    certs = {}
    for name, issuer, ca, path_length, eku in [
        ('root', 'root', True, 1, None),
        ('intermediate', 'root', True, 0, None),
        ('server', 'intermediate', False, None, ExtendedKeyUsageOID.SERVER_AUTH),
        ('client', 'intermediate', False, None, ExtendedKeyUsageOID.CLIENT_AUTH),
    ]:
        end = now - timedelta(days=1) if expired and name == 'server' else now + timedelta(days=7)
        builder = (x509.CertificateBuilder().subject_name(names[name]).issuer_name(names[issuer])
                   .public_key(keys[name].public_key()).serial_number(x509.random_serial_number())
                   .not_valid_before(now - timedelta(days=2)).not_valid_after(end)
                   .add_extension(x509.BasicConstraints(ca=ca, path_length=path_length), critical=True)
                   .add_extension(x509.KeyUsage(digital_signature=True, content_commitment=False,
                       key_encipherment=False, data_encipherment=False, key_agreement=False,
                       key_cert_sign=ca, crl_sign=ca, encipher_only=False, decipher_only=False), critical=True)
                   .add_extension(x509.SubjectKeyIdentifier.from_public_key(keys[name].public_key()), False)
                   .add_extension(x509.AuthorityKeyIdentifier.from_issuer_public_key(keys[issuer].public_key()), False))
        if eku:
            builder = builder.add_extension(x509.ExtendedKeyUsage([eku]), False)
            builder = builder.add_extension(x509.SubjectAlternativeName([
                x509.DNSName('invoice.test' if name == 'server' else 'client.test')]), False)
        certs[name] = builder.sign(keys[issuer], hashes.SHA256())
    return keys, certs


def tls_trial(hostname='invoice.test', trust_root=True, expired=False,
              include_intermediate=True, mtls=False, send_client=True,
              client_wrong_eku=False):
    """Handshake and exchange application bytes. Failures raise ssl.SSLError.

    Private PEM files are disposable teaching keys in a temporary directory.
    Does not implement online revocation, networking, or authorization policy.
    """
    keys, certs = make_pki(expired)
    pem = lambda c: c.public_bytes(serialization.Encoding.PEM)
    with tempfile.TemporaryDirectory(prefix='workshop-pki-') as directory:
        base = Path(directory)
        for name in ('server', 'client'):
            selected = 'server' if name == 'client' and client_wrong_eku else name
            chain = pem(certs[selected])
            if include_intermediate or name == 'client':
                chain += pem(certs['intermediate'])
            (base / (name + '.pem')).write_bytes(chain)
            (base / (name + '.key')).write_bytes(keys[selected].private_bytes(
                serialization.Encoding.PEM, serialization.PrivateFormat.PKCS8,
                serialization.NoEncryption()))
        server_context = ssl.SSLContext(ssl.PROTOCOL_TLS_SERVER)
        client_context = ssl.SSLContext(ssl.PROTOCOL_TLS_CLIENT)
        for context in (server_context, client_context):
            context.minimum_version = context.maximum_version = ssl.TLSVersion.TLSv1_3
        server_context.load_cert_chain(str(base / 'server.pem'), str(base / 'server.key'))
        if trust_root:
            client_context.load_verify_locations(cadata=pem(certs['root']).decode())
        if mtls:
            server_context.verify_mode = ssl.CERT_REQUIRED
            server_context.load_verify_locations(cadata=pem(certs['root']).decode())
        if send_client:
            client_context.load_cert_chain(str(base / 'client.pem'), str(base / 'client.key'))
        ci, co, si, so = (ssl.MemoryBIO() for _ in range(4))
        client = client_context.wrap_bio(ci, co, server_hostname=hostname)
        server = server_context.wrap_bio(si, so, server_side=True)
        completed = [False, False]

        def transfer():
            for outgoing, incoming in ((co, si), (so, ci)):
                if outgoing.pending:
                    incoming.write(outgoing.read())

        for _ in range(100):
            for index, peer in enumerate((client, server)):
                if not completed[index]:
                    try:
                        peer.do_handshake()
                        completed[index] = True
                    except ssl.SSLWantReadError:
                        pass
            transfer()
            if all(completed):
                break
        else:
            raise RuntimeError('TLS handshake stalled')
        payload = b'synthetic confidential invoice'
        client.write(payload)
        transfer()
        assert server.read(4096) == payload
        return {'version': client.version(), 'cipher': client.cipher()[0],
                'client_authenticated': bool(server.getpeercert()),
                'application_bytes': len(payload)}


def expect_rejection(operation, exceptions):
    """Assert the negative case, without accepting a silent failure."""
    try:
        operation()
    except exceptions:
        return
    raise AssertionError('Expected rejection did not occur')


def derive_day2(secret, transcript, direction=b'alice-to-bob'):
    """Teaching KDF only, not a standardized TLS or hybrid key schedule."""
    return HKDF(algorithm=hashes.SHA256(), length=32, salt=None,
                info=b'workshop-day2:v1|' + hashlib.sha256(transcript).digest()
                + b'|' + direction).derive(secret)


# Lab 6: ML-DSA Signatures

**30 minutes guided · 60–75 minutes independently.** Notebook (see course website) · Instructor (see course website) · Solutions (see course website)

## Goal and preparation

Implement fail-closed ML-DSA verification with a trusted key and purpose context. Measure signature size and distinguish signature validity from release authorization. Complete Sessions 5 and 11 and use Day 2 setup (see course website). Only synthetic release bytes are used; nothing is installed.



```mermaid
flowchart LR
    M["Release bytes and context"] --> S["ML-DSA signature"]
    S --> V["Trusted key verification"]
    V --> P["Product, version and authorization checks"]
```

Verification does not replace the final policy box.

## Task one: sign and measure — 5 minutes


In [ ]:
from cryptography.hazmat.primitives.asymmetric.mldsa import MLDSA65PrivateKey
from cryptography.exceptions import InvalidSignature
signer = MLDSA65PrivateKey.generate()
trusted_key = signer.public_key()
message = b'firmware:v1|product=training-device|version=9|digest=synthetic'
context = b'lab6-release'
signature = signer.sign(message, context)
trusted_key.verify(signature, message, context)
assert len(signature) == 3309
print('PASS: generated ML-DSA-65 signature; bytes:', len(signature))


## Task two: implement the verifier — 15 minutes

Implement the learner function to return `True` only on successful verification, and `False` on `InvalidSignature`. It receives an already trusted public key. Do not replace it with a key from the signed bundle, alter bytes, ignore context, or catch every exception as success.


In [ ]:
def learner_verify(public, message, signature, context):
    raise NotImplementedError('Verify exact bytes and context')

def check_verifier(candidate):
    assert candidate(trusted_key, message, signature, context) is True
    assert candidate(trusted_key, message + b'!', signature, context) is False
    assert candidate(trusted_key, message, signature, b'wrong-purpose') is False
    assert candidate(trusted_key, message, signature[:-1], context) is False
    other = MLDSA65PrivateKey.generate().public_key()
    assert candidate(other, message, signature, context) is False

try:
    check_verifier(learner_verify)
except NotImplementedError:
    print('NOT ATTEMPTED: learner ML-DSA verifier')
else:
    print('PASS: learner ML-DSA verifier')


<details><summary>Hints</summary><p>The library returns None on success and raises InvalidSignature for the expected cryptographic mismatch. Pass signature, message and context in that order. Return an explicit Boolean; do not rely on the truthiness of verify's return value.</p></details>

## Reference, controls and debrief — 10 minutes


In [ ]:
def reference_verify(public, message, signature, context):
    try:
        public.verify(signature, message, context)
    except InvalidSignature:
        return False
    return True
check_verifier(reference_verify)
print('PASS: supplied Lab 6 reference checks')

def observe_signature(case='valid'):
    if case == 'changed message':
        return reference_verify(trusted_key, message + b'!', signature, context)
    if case == 'changed context':
        return reference_verify(trusted_key, message, signature, b'other')
    if case == 'truncated':
        return reference_verify(trusted_key, message, signature[:-1], context)
    return reference_verify(trusted_key, message, signature, context)
assert observe_signature() and not observe_signature('changed context')
if 'get_ipython' in globals():
    import ipywidgets as widgets
    from IPython.display import display
    display(widgets.interactive(observe_signature, case=['valid', 'changed message', 'changed context', 'truncated']))


Use `observe_signature('changed message')` without widgets. Explain why a trusted signer can still sign the wrong product and why a replayed intact signature still verifies. Reuse Session 5's acceptance-policy reasoning; do not parse and install arbitrary signed bytes.

Submit your verifier, failure table, measured signature/public-key sizes, and a proposal for authenticating a replacement verification key to an offline device. Reference checks are not learner completion. Extension: test an AND versus OR acceptance policy for two signature families and describe the downgrade implications. See solutions (see course website).


In [ ]:
print("PASS: completed lab-06-pqc-signatures demonstrations; learner status is reported separately")
